# Formatting Envision files

We will be formatting the envision outputs

In [2]:
import polars as pl
from pathlib import Path
from datetime import datetime
import io
import numpy as np

# raw_envision_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/EnVision_2026-04-01._.RAW_DATA")
raw_envision_path = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/Derfel_Partch_20260511_EnVision._.RAW_DATA")

data_headers = ('Plate', 'Barcode', 'Well')
end_col = 6

In [15]:
def parse_envision(path: Path):
    with open(path, 'r') as f:
        lines = [l.strip() for l in f.readlines()]
    
    header_idx = None
    
    for i, line in enumerate(lines):
        line = line.strip()
        cols = [c.strip() for c in line.split(",")]
        if data_headers[0] in cols and data_headers[1] in cols and data_headers[2] in cols:
            header_idx = i
            break
    
    data_lines = [lines[header_idx]]
    
    for line in lines[header_idx + 1:]:
        if not line.strip():
            break
        first_col = line.split(',')[0].strip()
        if not first_col.isdigit():
            break
        data_lines.append(line)
        
    cleaned = "\n".join(data_lines).replace('=""', '')
    
    return pl.read_csv(io.StringIO(cleaned), null_values=['', 'NA'])

def get_measurement_date(path: Path) -> datetime:
    with open(path, 'r') as f:
        lines = f.readlines()
    header = [c.strip() for c in lines[1].split(',')]
    values = [c.strip().replace('=""', '') for c in lines[2].split(',')]
    
    row = dict(zip(header, values))
    
    return datetime.strptime(row['Measurement date'], '%m/%d/%Y %I:%M:%S %p')
    
        

let's compute real elapsed timepoints from the data

In [16]:
files = sorted(raw_envision_path.iterdir())
file_dates = [(f, get_measurement_date(f)) for f in files]
file_dates.sort(key=lambda x: x[1])

t0 = file_dates[0][1]
timepoints = [(f, round((dt - t0).total_seconds() / 3600 * 2) / 2) for f, dt in file_dates]
print(timepoints)

[(PosixPath('/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/Derfel_Partch_20260511_EnVision._.RAW_DATA/3331.csv'), 0.0), (PosixPath('/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/Derfel_Partch_20260511_EnVision._.RAW_DATA/3332.csv'), 0.5), (PosixPath('/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/Derfel_Partch_20260511_EnVision._.RAW_DATA/3333.csv'), 1.0), (PosixPath('/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/Derfel_Partch_20260511_EnVision._.RAW_DATA/3334.csv'), 1.5), (PosixPath('/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/Derfel_Partch_20260511_EnVision._.RAW_DATA/3335.csv'), 2.0), (PosixPath('/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/Derfel_Partch_20260511_EnVision._.RAW_DATA/3336.csv'), 2.5), (PosixPath('/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/Derfel_Partch_20260511_EnVision._.RAW_DATA/3337.csv'), 3.0), (PosixPath('/home/d

In [17]:
# we will be using actual timepoints in this experiment (not using the sequential ints)

dfs = []

for f, time in timepoints:
    df = (
        parse_envision(f)
        .drop(
            ["MeasTime", 
             "CalcResultI_duplicated_0",
             "Signal_duplicated_0",
             "MeasTime_duplicated_0",
             "_duplicated_0",
             "_duplicated_1",
             "",
             "Barcode", 
             "Flashes/Time",
             "Plate"])
        .with_columns(pl.lit(time).alias("Timepoint"))
    )
    dfs.append(df)
    
final_formatted_df = pl.concat(dfs)

display(final_formatted_df)
final_formatted_df.write_csv("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/enVision/enVision_formatted/2026-05-11._.FORMATTED.tsv", separator="\t")

Well,CalcResultI,Signal,Timepoint
str,i64,i64,f64
"""A01""",20,4,0.0
"""A02""",20,4,0.0
"""A03""",20,4,0.0
"""A04""",40,8,0.0
"""A05""",40,8,0.0
…,…,…,…
"""P20""",40,8,138.5
"""P21""",20,4,138.5
"""P22""",20,4,138.5


Let's find the missing time points

In [13]:
expected_interval = 0.5
gaps = []

for i in range(1, len(timepoints)):
    prev_t = timepoints[i - 1][1]
    curr_t = timepoints[i][1]
    diff =  curr_t - prev_t
    if diff > expected_interval:
        missing = list(np.arange(prev_t + expected_interval, curr_t, expected_interval))
        gaps.append({
            'after_file'      : timepoints[i-1][0].stem,
            'before_file'     : timepoints[i][0].stem,
            'after_hour'      : prev_t,
            'before_hour'     : curr_t,
            'missing_hours'   : missing,
            'n_missing'       : len(missing),
        })

# report
if gaps:
    print(f"Found {len(gaps)} gap(s):")
    for g in gaps:
        print(f"  between file {g['after_file']} ({g['after_hour']}h) "
              f"and {g['before_file']} ({g['before_hour']}h) "
              f"— missing timepoints: {g['missing_hours']}")
else:
    print("No gaps found")

No gaps found


# This is for formatting platemaps

In [8]:
# we have different timings when dex was applied to certain wells
platemap_dir = Path("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/platemaps/2026-05-11")

dfs = []
for file in platemap_dir.iterdir():
    df = pl.read_excel(file, engine='openpyxl').with_columns(pl.col('Source_Concentration').cast(pl.String))
    dfs.append(df)
    
dfs = pl.concat(dfs)
dfs = dfs.with_columns(pl.col("strain").str.to_lowercase())
display(dfs)
# print(dfs['384_Well'].unique())

dfs.write_excel("/home/derfelt/PartchLabFiles/Circadian Rhythms/Live Study/platemaps/2026-05-11Echo_transfer.xlsx")

Source_Plate_Name,UCSC_CSC_plate_ID,Source_Well,Source_Concentration,Source_Concentration_Units,Destination_Plate_Name,Destination_Plate_Barcode,384_Well,MoleculeID,Human_Readable_Name,Transfer_Volume_nL,Assay_Volume_ul,Concentration,Units,Well_X,Well_Y,ScoringClass,CompoundClass,sample_type,strain
str,null,str,str,str,str,null,str,str,str,f64,i64,f64,str,i64,i64,null,null,str,str
"""P001""",null,"""A01""",null,null,"""2026-05-05 384Corning3570""",null,"""P24""","""food coloring""",null,500.0,30,null,null,16,24,null,null,null,null
"""McClain""",null,"""C03""","""10""","""mM""","""2026-05-05 384Corning3570""",null,"""F04""","""MK-8692 25uM""",null,75.0,30,null,null,6,4,null,null,"""dex""","""bmal1"""
"""McClain""",null,"""C03""","""10""","""mM""","""2026-05-05 384Corning3570""",null,"""F05""","""MK-8692 25uM""",null,75.0,30,null,null,6,5,null,null,"""dex""","""bmal1"""
"""McClain""",null,"""C03""","""10""","""mM""","""2026-05-05 384Corning3570""",null,"""F06""","""MK-8692 25uM""",null,75.0,30,null,null,6,6,null,null,"""dex""","""bmal1"""
"""McClain""",null,"""C03""","""10""","""mM""","""2026-05-05 384Corning3570""",null,"""F07""","""MK-8692 12.5uM""",null,37.5,30,null,null,6,7,null,null,"""dex""","""bmal1"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""P001""",null,"""J01""","""1""","""mM""","""2026-05-05_384PV_Ch""",null,"""H18""","""Dex_Chroma_bmal1""","""Dex_6TP""",2.5,30,0.083,"""uM""",8,18,null,null,null,"""bmal1"""
"""P001""",null,"""J01""","""1""","""mM""","""2026-05-05_384PV_Ch""",null,"""I16""","""Dex_PhenoVue_per2""","""Dex_6TP""",2.5,30,0.083,"""uM""",9,16,null,null,null,"""per2"""
"""P001""",null,"""J01""","""1""","""mM""","""2026-05-05_384PV_Ch""",null,"""I17""","""Dex_PhenoVue_per2""","""Dex_6TP""",2.5,30,0.083,"""uM""",9,17,null,null,null,"""per2"""
